In [1]:
!pip -q install transformers sentence-transformers faiss-cpu PyPDF2 accelerate

In [2]:
import numpy as np
import faiss
import torch

from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [3]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

In [4]:
def generate_answer(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
pdf_path = "/kaggle/input/datasets/mariamessam47/data123/THU.pdf"

In [6]:
reader = PdfReader(pdf_path)
text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:500])

1 . General Overview 
Tips Hindawi University (THU) is a premier institution of higher education located in the 
heart of the Middle 
East. Founded in 1963, the university has grown into a globally recognized center for 
academic excellence 
and innovation. With over six decades of educational leadership, THU has produced more 
than 150,000 
graduates who serve in diverse industries and acad emic circles worldwide  . 
The university is accredited by the International Commission for Academic Stan


In [7]:
def chunk_text(text,size=100):

    words=text.split()

    chunks=[]

    for i in range(0,len(words),size):

        chunks.append(" ".join(words[i:i+size]))

    return chunks

In [8]:
chunks=chunk_text(text)

In [9]:
embedding_model=SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings=embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
dimension=embeddings.shape[1]

index=faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [11]:
def retrieve(question,k=3):

    q_embedding=embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    D,I=index.search(q_embedding,k)

    return [chunks[i] for i in I[0]]

In [12]:
questions=[

"Where is Tips Hindawi University located?",

"Does the university offer online programs?",

"Is there financial aid for international students?",

"What languages are used for instruction?"

]

In [13]:
def generate_answer(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True          # <-- مهم
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return answer

In [15]:
for question in questions:
    context = "\n".join(retrieve(question))
    prompt = f"""
Answer ONLY using the following context.
Context:
{context}
Question:
{question}
If the answer is not in the context, reply:
The document does not mention this information.
"""
    # Generate the answer
    answer = generate_answer(prompt)
    print("=" * 70)
    print("Question:", question)
    print("Answer:")
    print(answer)
    print()

Question: Where is Tips Hindawi University located?
Answer:
The main campus of Tips Hindawi University is located in the capital city.

Question: Does the university offer online programs?
Answer:
The document does not mention this information.

Question: Is there financial aid for international students?
Answer:
The document mentions "Need-Based Grants" under section 4.3 Scholarships, which could potentially include financial aid for international students based on need. However, it does not explicitly state that international students are eligible for these grants. Therefore, the document does not provide a definitive answer on whether financial aid is available specifically for international students.

Question: What languages are used for instruction?
Answer:
The document does not mention this information.

